# OPTUNA HYPERPARAMETER TUNING

### Imports

In [1]:
#sklearn
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, cohen_kappa_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.base import clone

# Metrics
from sklearn.metrics import f1_score, make_scorer

# Imbalanced-learn (SMOTE, Oversampling)
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.pipeline import make_pipeline

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optuna
import optuna

# Preprocessing
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessorLOEEALL

import numpy as np
import pandas as pd

#Warnings lightGBM
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)


c:\Users\asier.gomez\AppData\Local\anaconda3\envs\petfinder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset loading

In [2]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

categorical_cols = ['Type', 'Gender', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'RescuerID', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength','Breed1', 'Breed2', 'State']

(10045, 32)
(10045,)
(4948, 32)
(4948,)


### Initialization

In [3]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializng the second evaluation technique
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
scoring = {
        "f1_macro": "f1_macro",
        "QWK": qwk
    }

### XGBoost hyperparameter tuning

In [ ]:
"""
def objective_xgb(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.05)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)
    
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss"    # for multiple classification
    }

    #Creamos pipeline con preprocessor y SMOTE
    pipe_base = imb_make_pipeline(
        preprocessorLOEEALL,
        RandomOverSampler(random_state=42),      
        XGBClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
xgb_study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study2",
    storage="sqlite:///optuna_studies/xgb_study.db",  # se guarda en este archivo
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

#we enter the parameters with the best result in the last study
#If it is the first study we put the default parameters
xgb_study.enqueue_trial({
    "looe_sigma": 0.028238316352854904,
    "n_estimators": 976,
    "max_depth": 9,
    "learning_rate": 0.014031401130172998,
    "min_child_weight": 6,
    "subsample": 0.7395366759188357,
    "colsample_bytree": 0.8063729967181755
})

xgb_study.optimize(objective_xgb, n_trials=100)

print(f"Mejores parámetros: {xgb_study.best_params}")
"""

[I 2026-03-14 07:49:57,923] A new study created in RDB with name: xgb_study2
[I 2026-03-14 07:51:06,602] Trial 0 finished with value: 0.4176029298921349 and parameters: {'looe_sigma': 0.028238316352854904, 'n_estimators': 976, 'max_depth': 9, 'learning_rate': 0.014031401130172998, 'subsample': 0.7395366759188357, 'colsample_bytree': 0.8063729967181755, 'min_child_weight': 6}. Best is trial 0 with value: 0.4176029298921349.
[I 2026-03-14 07:51:23,413] Trial 1 finished with value: 0.37407660847475077 and parameters: {'looe_sigma': 0.008935538094064511, 'n_estimators': 486, 'max_depth': 5, 'learning_rate': 0.19446566339847182, 'subsample': 0.8282609244084409, 'colsample_bytree': 0.8031070919429738, 'min_child_weight': 1}. Best is trial 0 with value: 0.4176029298921349.
[I 2026-03-14 07:52:25,611] Trial 2 finished with value: 0.40661420869910553 and parameters: {'looe_sigma': 0.010802489415310224, 'n_estimators': 1094, 'max_depth': 10, 'learning_rate': 0.0372574328386415, 'subsample': 0.77

Mejores parámetros: {'looe_sigma': 0.010773764065091765, 'n_estimators': 552, 'max_depth': 10, 'learning_rate': 0.01294031155889707, 'subsample': 0.6459278279430541, 'colsample_bytree': 0.6169935895547909, 'min_child_weight': 1}


### CatBoost hyperparameter tuning

In [ ]:
"""
def objective_catboost(trial):
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "random_strength": 1,
        "bagging_temperature": 1,
        "border_count": 128,
        "verbose":0, 
        "cat_features": categorical_cols,
        "allow_writing_files":False,
        "random_state": 42,
        "thread_count": -1  
    }

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(X_fold_train, y_fold_train)
        preds = model.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
cat_study = optuna.create_study(
    direction="maximize",
    study_name="cat_study_native",
    storage="sqlite:///optuna_studies/cat_study.db",  # se guarda en este archivo
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

cat_study.enqueue_trial({
    "n_estimators": 1000,         
    "max_depth": 6,               
    "learning_rate": 0.03,        
    "l2_leaf_reg": 3.0,
})

cat_study.optimize(objective_catboost, n_trials=100)

print(f"Mejores parámetros: {cat_study.best_params}")
"""

[I 2026-03-13 23:46:37,081] Using an existing study with name 'cat_study_native' instead of creating a new one.
[I 2026-03-13 23:54:37,610] Trial 1 finished with value: 0.4433665335151252 and parameters: {'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-13 23:58:46,230] Trial 2 finished with value: 0.43588281142792384 and parameters: {'n_estimators': 657, 'max_depth': 5, 'learning_rate': 0.12889699430870627, 'l2_leaf_reg': 2.971221171040712}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-14 00:02:04,109] Trial 3 finished with value: 0.436811080806741 and parameters: {'n_estimators': 421, 'max_depth': 6, 'learning_rate': 0.11785770555979118, 'l2_leaf_reg': 6.451872448595916}. Best is trial 1 with value: 0.4433665335151252.
[I 2026-03-14 00:05:38,682] Trial 4 finished with value: 0.43909749153721833 and parameters: {'n_estimators': 465, 'max_depth': 6, 'learning_rate': 0.0947608269863

Mejores parámetros: {'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}


### RandomForest hyperparameter tuning

In [ ]:
"""
def objective_rf(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.06)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_int("max_depth", 10, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "bootstrap": True,
        "class_weight": "balanced", 
        "random_state": 42,
        "n_jobs": -1
    }

    pipe_base = make_pipeline(
        preprocessorLOEEALL,
        RandomForestClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)

# Crear estudio de Optuna
rf_study = optuna.create_study(
    direction="maximize",
    study_name="rf_study2",
    storage="sqlite:///optuna_studies/rf_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

#We use the best hyperparameters of the previous study
#If it is the first study we use the default parameters
rf_study.enqueue_trial({
    "looe_sigma": 0.013264638423263085,
    "n_estimators": 505,
    "max_depth": 60,
    "min_samples_split": 11,
    "min_samples_leaf": 1,
    "max_features": "sqrt"
})

# Lanzar optimización
rf_study.optimize(objective_rf, n_trials=100)

# Resultados
print(f"Mejores parámetros: {rf_study.best_params}")
"""

[I 2026-03-13 18:30:45,042] A new study created in RDB with name: rf_study2
[I 2026-03-13 18:30:56,111] Trial 0 finished with value: 0.41728014610265063 and parameters: {'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:07,591] Trial 1 finished with value: 0.40168592536987396 and parameters: {'looe_sigma': 0.03883443450450948, 'n_estimators': 508, 'max_depth': 21, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:11,808] Trial 2 finished with value: 0.39955385872732274 and parameters: {'looe_sigma': 0.037987927251354234, 'n_estimators': 144, 'max_depth': 71, 'min_samples_split': 20, 'min_samples_leaf': 13, 'max_features': 'log2'}. Best is trial 0 with value: 0.41728014610265063.
[I 2026-03-13 18:31:18,581] Trial 3 finished with valu

Mejores parámetros: {'looe_sigma': 0.013264638423263085, 'n_estimators': 505, 'max_depth': 60, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


### LightGBM hyperparameter tuning

In [ ]:
"""
def objective_lgbm_balanced(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.1)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7 , 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }

    # num_leaves <= 2^(max_depth) - 1
    # Forzamos a que si Optuna elige una mala combinación, se corrija sola:
    max_leaves_allowed = (2 ** params["max_depth"]) - 1
    params["num_leaves"] = min(params["num_leaves"], max_leaves_allowed)

    # Creamos pipeline clásico de Scikit-Learn (sin SMOTE)
    pipe_base = imb_make_pipeline(
        preprocessorLOEEALL,
        RandomOverSampler(random_state = 42),
        LGBMClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)


# Crear estudio de Optuna (con nombre nuevo para no pisar el anterior)
lgbm_study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_study1",
    storage="sqlite:///optuna_studies/lgbm_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

# Trial 0 inicial razonable
lgbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,          
    "max_depth": 15,              
    "num_leaves": 31,             
    "learning_rate": 0.1,        
    "min_child_samples": 20,      
    "subsample": 1.0,             
    "colsample_bytree": 1.0       
})

# Lanzar optimización
lgbm_study.optimize(objective_lgbm_balanced, n_trials=100)

# Resultados
print(f"Mejores parámetros: {lgbm_study.best_params}")
"""

[I 2026-03-13 18:46:46,523] A new study created in RDB with name: lgbm_study1
[I 2026-03-13 18:46:51,427] Trial 0 finished with value: 0.39194816689722767 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 15, 'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 1.0}. Best is trial 0 with value: 0.39194816689722767.
[I 2026-03-13 18:47:02,920] Trial 1 finished with value: 0.3799719119654584 and parameters: {'looe_sigma': 0.00784130618090908, 'n_estimators': 161, 'max_depth': 12, 'learning_rate': 0.2645936406421905, 'num_leaves': 56, 'min_child_samples': 10, 'subsample': 0.9938652830131463, 'colsample_bytree': 0.6241313538255718}. Best is trial 0 with value: 0.39194816689722767.
[I 2026-03-13 18:47:43,562] Trial 2 finished with value: 0.37859275945447596 and parameters: {'looe_sigma': 0.010098478722223747, 'n_estimators': 650, 'max_depth': 11, 'learning_rate': 0.17283581985646962, 'num_leaves': 91, 'min_child_samples'

Mejores parámetros: {'looe_sigma': 0.017274353532403468, 'n_estimators': 431, 'max_depth': 14, 'learning_rate': 0.021117244921237944, 'num_leaves': 49, 'min_child_samples': 8, 'subsample': 0.7392171099865767, 'colsample_bytree': 0.9999202226086774}


### Gradient Boosting hyperparameter tuning

In [ ]:
"""
def objective_gbm(trial):
    # 1. Rango ajustado para tu target multiclase (0 a 4)
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)
    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 6), 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0), 
        "random_state": 42
    }

    # 3. Pipeline normal (make_pipeline) SIN técnicas de balanceo
    pipe_base = make_pipeline(
        preprocessorLOEEALL,
        GradientBoostingClassifier(**params)
    )

    fold_scores = []

    for step, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        pipe = clone(pipe_base)

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        pipe.fit(X_fold_train, y_fold_train)
        preds = pipe.predict(X_fold_val)

        qwk = cohen_kappa_score(y_fold_val, preds, weights="quadratic")

        fold_scores.append(qwk)

        trial.report(qwk, step)

        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(fold_scores)


# Crear estudio de Optuna
gbm_study = optuna.create_study(
    direction="maximize",
    study_name="gbm_study",
    storage="sqlite:///optuna_studies/gbm_study.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

# Valores por defecto reales de GradientBoostingClassifier
gbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.1,
    "min_samples_leaf": 1,
    "subsample": 1.0,
    "max_features": 1.0
})

# Lanzar optimización
gbm_study.optimize(objective_gbm, n_trials=100)

# Resultados
print(f"Mejores parámetros: {gbm_study.best_params}")
"""

[I 2026-03-13 20:30:38,864] A new study created in RDB with name: gbm_study
[I 2026-03-13 20:32:15,581] Trial 0 finished with value: 0.3994245456952566 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'min_samples_leaf': 1, 'subsample': 1.0, 'max_features': 1.0}. Best is trial 0 with value: 0.3994245456952566.
[I 2026-03-13 20:33:54,969] Trial 1 finished with value: 0.3300455048466535 and parameters: {'looe_sigma': 0.2012208862374252, 'n_estimators': 102, 'max_depth': 5, 'learning_rate': 0.11339828749227336, 'min_samples_leaf': 10, 'subsample': 0.8116584059118779, 'max_features': 0.7561180571627557}. Best is trial 0 with value: 0.3994245456952566.
[I 2026-03-13 20:38:07,122] Trial 2 finished with value: 0.39042108115938745 and parameters: {'looe_sigma': 0.06135296511275576, 'n_estimators': 314, 'max_depth': 6, 'learning_rate': 0.03964487902594386, 'min_samples_leaf': 44, 'subsample': 0.6233148013999327, 'max_features': 0.7262514297438959}.

Mejores parámetros: {'looe_sigma': 0.021372534159264083, 'n_estimators': 138, 'max_depth': 3, 'learning_rate': 0.0788176779144456, 'min_samples_leaf': 32, 'subsample': 0.9993202620699826, 'max_features': 0.7944324429031259}


### Voting and stacking classifiers

In [ ]:
rf_study = optuna.load_study(
    study_name="rf_study2", 
    storage="sqlite:///optuna_studies/rf_study.db" 
)

lgbm_study = optuna.load_study(
    study_name="lgbm_study1", 
    storage="sqlite:///optuna_studies/lgbm_study.db" 
)


cat_study = optuna.load_study(
    study_name="cat_study_native", 
    storage="sqlite:///optuna_studies/cat_study.db"
)

xgb_study = optuna.load_study(
    study_name="xgb_study2", 
    storage="sqlite:///optuna_studies/xgb_study.db")

gbm_study = optuna.load_study(
    study_name="gbm_study",
    storage="sqlite:///optuna_studies/gbm_study.db" 
)

In [23]:
rf_params = rf_study.best_params.copy()
loe_sigma_rf = rf_params.pop("looe_sigma", None)

rf_best = RandomForestClassifier(
    **rf_params,    
    bootstrap=True,            
    class_weight="balanced",    
    random_state=42,
    n_jobs=-1
)

cat_best = CatBoostClassifier(
    **cat_study.best_params,     
    random_strength=1,            
    bagging_temperature=1,
    border_count=128,
    verbose=0,
    allow_writing_files=False,
    random_state=42,
    thread_count=-1
)

xgb_params = xgb_study.best_params.copy()
loe_sigma_xgb = xgb_params.pop("looe_sigma", None)

xgb_best = XGBClassifier(
    **xgb_params,
    random_state=42,         
    n_jobs=-1,
    eval_metric="mlogloss"  
)

lgbm_params = lgbm_study.best_params.copy()
loe_sigma_lgbm = lgbm_params.pop("looe_sigma", None)

lgbm_best = LGBMClassifier(
    **lgbm_params,  
    subsample_freq=1,          
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

gbm_params = gbm_study.best_params.copy()
loe_sigma_gbm = gbm_params.pop("looe_sigma", None)

gbm_best = GradientBoostingClassifier(
    **gbm_params, 
    random_state=42          
)

In [ ]:
"""
rf_best = RandomForestClassifier(n_estimators =  573, max_depth = None, max_features = 'log2', random_state=42, n_jobs=-1)
lgbm_best = LGBMClassifier(n_estimators = 679, max_depth = 13, learning_rate = 0.021354407760849407, num_leaves = 110, min_child_samples = 13, subsample = 0.937569357477504, colsample_bytree = 0.6361451774580816, colsample_bytree = 0.6321534978115201, random_state=42, n_jobs=-1, verbose=-1)
cat_best = CatBoostClassifier(**cat_study.best_params, random_state=42, verbose=0)
xgb_best = XGBClassifier(**xgb_study.best_params, random_state=42, n_jobs=-1)
"""

In [27]:
# 1. Random Forest (BorderlineSMOTE)
pipe_rf = imb_make_pipeline(
    clone(preprocessorLOEEALL), 
    RandomOverSampler(random_state=42), 
    rf_best
)

# 2. LightGBM (Balanced interno, sin oversampling externo)
pipe_lgbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42), 
    lgbm_best
)

# 3. Gradient Boosting (RandomOverSampler)
pipe_gbm = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    gbm_best
)

# 4. CatBoost (Nativo y balanced, sin preprocesador)
pipe_cat = imb_make_pipeline(
    cat_best
)

pipe_xgb = imb_make_pipeline(
    clone(preprocessorLOEEALL),
    RandomOverSampler(random_state=42),
    xgb_best
)

In [29]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', pipe_cat),
        ('xgb', pipe_xgb),
        ('gbm', pipe_gbm)
    ],
    voting='soft',   # TRUCO 1: 'soft' suele dar mejor métrica que 'hard'
    #weights=[1, 2, 2, 1] # TRUCO 2: Ponderación (opcional)
)

parametros_entrenamiento = {
    'cat__cat_features': list(categorical_cols)
}

results_cv = cross_validate(voting_clf, X_train, y_train, cv=skf, scoring=scoring, params=parametros_entrenamiento)

print(f"QWK del Ensemble: {np.mean(results_cv['test_QWK']):.4f}")

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\asier.gomez\AppData\Local\anaconda3\envs\petfinder\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\asier.gomez\AppData\Local\anaconda3\envs\petfinder\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\asier.gomez\AppData\Local\anaconda3\envs\petfinder\Lib\site-packages\sklearn\ensemble\_voting.py", line 382, in fit
    _raise_for_params(fit_params, self, "fit", allow=["sample_weight"])
  File "c:\Us

In [ ]:
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', pipe_cat),
        ('xgb', pipe_xgb)
    ],
    # TRUCO 1: El Meta-Learner debe ser un modelo simple
    final_estimator=LogisticRegression(max_iter=1000, random_state=42), 
    
    cv=skf, 
    
    # TRUCO 3: 'passthrough=False' (por defecto) para evitar overfitting
    passthrough=False,
    
    n_jobs=-1 # Para no colapsar la RAM
)

### Prueba CatBoost

In [ ]:
CB = CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
pipe = make_pipeline(CB)

#Cross-validation estratificada
fit_params = {'catboostclassifier__cat_features': categorical_cols}
score = cross_validate(pipe, X_train, y_train, cv=skf, params=fit_params, scoring=scoring)

print(np.mean(score['test_QWK']))

0.44089050835869587
